In [16]:
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
from meteostat import Point, Hourly, Stations
from tqdm import tqdm
import sys
import numpy as np

In [17]:
sys.path.append('Utils/')
from PlotUtils import setMplParam, getColour, getHistoParam 
# getHistoParam: 
# Nbins, binwidth, bins, counts, bin_centers  = 
# from ExternalFunctions import nice_string_output, add_text_to_ax
setMplParam()
from DataPipelineWorkShop import get_hourly_example, MeteoPreprocessor

In [ ]:
def make_single_window_dataframe(location: Point, 
                                 start_time: datetime, 
                                 window_hours: int,
                                 horizon_hours: int) -> pd.DataFrame:
    """
    Construct a minimal Meteostat dataframe covering exactly one forecasting window.

    Args:
        location (Point): Meteostat location object.
        start_time (datetime): End of the observation window (the forecast will start right after this).
        exp_ctx (ExperimentContext): Contains forecast.window and forecast.horizon.
    """

    # Fetch data covering just enough history for one forecast
    history_end   = start_time + pd.Timedelta(hours=horizon_hours)

    df = get_hourly_example(location, start_time, history_end)

    # Defensive timestamp conversion
    if isinstance(df.index, pd.PeriodIndex):
        df.index = df.index.to_timestamp()

    return df

In [28]:
kbh = Point(lat=55.6761, lon=12.5683)
start_time = datetime(2019, 6, 3, 0, 0)
df_true = make_single_window_dataframe(kbh, start_time, window_hours=24, horizon_hours=12)

In [29]:
df_true.columns

Index(['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres',
       'tsun', 'coco'],
      dtype='object')

In [30]:
df_true[['temp', 'rhum', 'prcp', 'wdir', 'wspd']]

,temp,rhum,prcp,wdir,wspd
time,,,,,
2019-06-03 00:00:00,16.0,71.0,0.0,130.0,25.9
2019-06-03 01:00:00,15.7,77.0,<NA>,140.0,27.7
2019-06-03 02:00:00,16.3,81.0,<NA>,140.0,22.3
2019-06-03 03:00:00,16.1,82.0,<NA>,150.0,16.6
2019-06-03 04:00:00,15.9,84.0,<NA>,150.0,11.2
2019-06-03 05:00:00,17.4,78.0,<NA>,200.0,11.2
2019-06-03 06:00:00,17.4,78.0,0.0,190.0,13.0
2019-06-03 07:00:00,18.9,74.0,<NA>,190.0,16.6
2019-06-03 08:00:00,20.5,71.0,<NA>,180.0,13.0


In [ ]:
base = "/Users/yhjo/Desktop/[2025] Be Resilient/Resilience/logs/"
prediction_files = {
    "b256_d64_h2_l4_win24_ho12_lr1e-4" : base+"20251110/141108/predictions/20251113_130918_epochepoch-19-valloss-0.1025.csv",
    "b256_d64_h2_l2_win24_ho12_lr1e-4" : base+"20251110/153321/predictions/20251113_131644_epochepoch-19-val_lossval_loss-0.1044.csv",
    "b512_d64_h2_l4_win24_ho12_lr1e-4" : base+"20251110/164812/predictions/20251113_132309_epochepoch-19-val_lossval_loss-0.1186.csv",
    "b512_d64_h2_l4_win24_ho12_lr3e-4" : base+"20251110/175859/predictions/20251113_132508_epochepoch-19-val_lossval_loss-0.0981.csv",
}

In [23]:
df_pred = pd.read_csv(prediction_files["b256_d64_h2_l4_win24_ho12"])
df_pred.columns

Index(['time', 'temp', 'rhum', 'prcp', 'wspd', 'sin_wdir', 'cos_wdir'], dtype='object')

In [24]:
processor = MeteoPreprocessor()
df_pred_inv = processor.inverse_transform(df_pred)

In [25]:
df_pred_inv

,time,temp,rhum,prcp,wspd,wdir
0,2019-06-03 00:00:00,13.769457,83.235651,0.021877,11.448321,204.952406
1,2019-06-03 01:00:00,14.292303,82.656369,0.012825,11.461506,206.955053
2,2019-06-03 02:00:00,13.421953,84.650454,0.013072,11.008427,215.998777
3,2019-06-03 03:00:00,13.523643,84.709540,0.002987,11.031719,218.716533
4,2019-06-03 04:00:00,13.608168,83.481392,-0.003813,11.377227,216.913380
5,2019-06-03 05:00:00,14.087631,82.520941,-0.011628,11.732664,215.207071
6,2019-06-03 06:00:00,14.701033,79.310396,-0.016595,12.996085,201.816883
7,2019-06-03 07:00:00,15.204529,77.170813,-0.027558,13.317376,206.139988
8,2019-06-03 08:00:00,15.800557,73.335299,-0.033467,14.302667,195.739123
9,2019-06-03 09:00:00,16.457846,70.005326,-0.030759,15.331733,192.898197


In [26]:
def plot_hourly_features(
    df_true: pd.DataFrame,
    df_pred: pd.DataFrame,
    features: list[str] = ['temp', 'rhum', 'prcp', 'wspd']
) -> None:
    df_true = df_true[:24]
    df_pred = df_pred[df_pred['window'] == 0]
    # --- normalise time index for df_true ---
    if 'time' in df_true.columns:
        df_true['time'] = pd.to_datetime(df_true['time'])
        df_true = df_true.set_index('time')
    elif isinstance(df_true.index, pd.PeriodIndex):
        df_true.index = df_true.index.to_timestamp()

    # --- normalise time index for df_pred ---
    if 'time' in df_pred.columns:
        df_pred['time'] = pd.to_datetime(df_pred['time'])
        df_pred = df_pred.set_index('time')
    elif isinstance(df_pred.index, pd.PeriodIndex):
        df_pred.index = df_pred.index.to_timestamp()

    n_features = len(features)
    fig, axes = plt.subplots(n_features, 1, figsize=(17, 3*n_features), sharex=True)
    if n_features == 1:
        axes = [axes]

    for ax, feature in zip(axes, features):
        df_true[feature].plot(ax=ax, label="True")
        df_pred[feature].plot(ax=ax, label="Predicted")
        ax.set_title(feature)
        ax.set_ylabel(feature)
        ax.legend()

    axes[-1].set_xlabel("Date")
    plt.tight_layout()
    plt.show()


plot_hourly_features(df_true, df_pred_inv, features=['temp', 'rhum', 'prcp', 'wspd'])

KeyError: 'window'